# 01. Limpieza y Preparación de Datos (ETL) — Supply Chain
Este notebook documenta la auditoría de calidad de datos, eliminación de información sensible (PII), tratamiento de valores nulos y duplicados, y estandarización de fechas para generar el dataset limpio `clean_supply.csv`.

In [ ]:
import pandas as pd
import numpy as np

# Carga del dataset original
df = pd.read_csv('../data/Original_Supply.csv', encoding='latin-1')
print(f'Dimensiones iniciales: {df.shape}')

In [ ]:
# Revisión de los datos
display(df.head(5))
df.info()
display(df.describe())

In [ ]:
# Auditoría de valores nulos
df.isnull().sum()[df.isnull().sum() > 0]

In [ ]:
# Eliminación de columnas sensibles (PII), vacías o redundantes
cols_to_drop = [
    'Customer Password', 'Customer Email', 'Customer Zipcode',
    'Latitude', 'Longitude', 'Order Item Profit Ratio',
    'Order Zipcode', 'Product Description', 'Product Image'
]
df = df.drop(columns=cols_to_drop, errors='ignore')

# Eliminación de registros duplicados
df = df.drop_duplicates()

# Conversión de variables de fecha a formato datetime
date_cols = ['order date (DateOrders)', 'shipping date (DateOrders)']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print(f'Dimensiones finales limpias: {df.shape}')

In [ ]:
# Manejo de outliers
def filtrar_outliers_iqr(df, sales):
    Q1 = df[sales].quantile(0.25)
    Q3 = df[sales].quantile(0.75)
    IQR = Q3 - Q1
    
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    
    # Retorna el DataFrame sin los outliers de esa columna
    return df[(df[sales] >= limite_inferior) & (df[sales] <= limite_superior)]

In [ ]:
# Exportación del dataset limpio listo para Business Intelligence en Excel
df.to_csv('../data/clean_supply.csv', index=False, encoding='utf-8')
print('Archivo clean_supply.csv exportado correctamente.')